# CSoT'26 — ML in Astronomy — Week 1 · Part 1: Foundations (Starter)

**Goal:** Set up Google Colab + a GPU runtime + PyTorch, create a tensor shaped like a galaxy image, and move it onto the GPU.

**Before you begin:**
1. Switch this notebook to a **GPU runtime**: `Runtime → Change runtime type → Hardware accelerator → GPU`.
2. Re-read [`09-project-task.md`](../09-project-task.md) if you haven't already.

Each `TODO` cell has a short instruction. Replace the placeholder with working code, then run the cell. If you get stuck, the concept markdowns in this folder cover everything you need.

**Do not** open `week1_solution.ipynb` until you've genuinely attempted every TODO here. Once you finish Part 1, move on to **Part 2** in `week1_data_starter.ipynb`.

## Step 1 — Confirm the GPU at the OS level

Run the shell command below. You should see a table mentioning an NVIDIA Tesla T4 (or similar). If you see `command not found`, your runtime is on CPU — fix it via `Runtime → Change runtime type → GPU`, then re-run.

In [ ]:
!nvidia-smi

Fri Jun  5 15:39:27 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   45C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

## Step 2 — Imports and version sanity check

Import `torch`, `torchvision`, and `matplotlib.pyplot as plt`. Print their versions. Colab pre-installs all three.

*Hint:* `torch.__version__`, `torchvision.__version__`, `matplotlib.__version__`.

In [ ]:
import torch
import torchvision
import matplotlib.pyplot as plt
import numpy as np

print(f'pytorch version : {torch.__version__}')
print(f"torchvision version: {torchvision.__version__}")
print(f"numpy version: {np.__version__}")

pytorch version : 2.11.0+cu128
torchvision version: 0.26.0+cu128
numpy version: 2.0.2


## Step 3 — Confirm the GPU from PyTorch

Print:
1. `torch.cuda.is_available()` — should be `True`.
2. `torch.cuda.device_count()` — should be at least `1`.
3. `torch.cuda.get_device_name(0)` — should be a real GPU name.

*Hint:* wrap (3) in an `if torch.cuda.is_available()` so it doesn't crash if you're (accidentally) on CPU.

In [ ]:
print(torch.cuda.is_available())
print(torch.cuda.device_count())
if torch.cuda.is_available():
    print("Device name    :", torch.cuda.get_device_name(0))
else:
    print("No GPU found")

True
1
Device name    : Tesla T4


## Step 4 — Define the `device` variable

The classic portable idiom: `device = "cuda" if torch.cuda.is_available() else "cpu"`. Print it.

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"
print(device)

cuda


## Step 5 — Create a 'toy galaxy' tensor on the CPU

Make a random tensor `x` of shape `(3, 64, 64)` using `torch.randn`. This is the same shape one galaxy image will have once we resize it next week (3 channels × 64 × 64 pixels).

Print `x.shape`, `x.dtype`, and `x.device`. You should see `cpu` for the device.

In [ ]:
x = torch.randn(3,64,64)
print(x.shape)
print(x.dtype)
print(x.device)

torch.Size([3, 64, 64])
torch.float32
cpu


## Step 6 — Move the tensor to the GPU

Use `.to(device)`. Save the result as `x_gpu`. Print `x_gpu.device`. You should see `cuda:0` (assuming the GPU is alive).

Then verify the shape is unchanged: `x_gpu.shape` should still be `torch.Size([3, 64, 64])`.

In [ ]:
x_gpu = x.to(device)
print(x_gpu.device)
print(x_gpu.shape)


cuda:0
torch.Size([3, 64, 64])


## Step 7 — Do something on the GPU

Take the first channel of `x_gpu` (`x_gpu[0]` → shape `(64, 64)`), matrix-multiply it with itself using `@`, and print the resulting tensor's shape and device. The shape should still be `(64, 64)` and the device should still be `cuda:0`.

In [ ]:
result = x_gpu[0] @ x_gpu[0]
print(result.shape)
print(result.device)

torch.Size([64, 64])
cuda:0


## Stretch Goal 1 — CPU vs GPU benchmark *(optional)*

Adapt the snippet from [`03-gpu-acceleration.md`](../03-gpu-acceleration.md) to time a matmul of two `(4096, 4096)` matrices on CPU vs GPU. Report the speedup. Don't forget `torch.cuda.synchronize()` before timing the GPU.

In [ ]:
import time

size = 4096
n_runs = 5

def bench(device_str):
    a = torch.randn(size, size, device=device_str)
    b = torch.randn(size, size, device=device_str)
    # Warm-up (especially important for GPU to avoid init overhead)
    for _ in range(2):
        (a @ b).sum().item()
    if device_str == "cuda":
        torch.cuda.synchronize()
    t0 = time.perf_counter()
    for _ in range(n_runs):
        c = a @ b
    if device_str == "cuda":
        torch.cuda.synchronize()
    t1 = time.perf_counter()
    return (t1 - t0) / n_runs

cpu_time = bench("cpu")
print(f"CPU average: {cpu_time*1000:.1f} ms")

if torch.cuda.is_available():
    gpu_time = bench("cuda")
    print(f"GPU average: {gpu_time*1000:.1f} ms")
    print(f"Speedup    : {cpu_time/gpu_time:.1f}x")

CPU average: 1441.4 ms
GPU average: 40.3 ms
Speedup    : 35.8x


## Stretch Goal 2 — Visualise a 'synthetic galaxy' *(optional)*

Use the cartoon construction from [`09-project-task.md`](../09-project-task.md) (Stretch Goal 3) to build a `(128, 128)` brightness map of a fake galaxy and display it with `plt.imshow`. Use a colormap like `inferno` or `magma`. Hide the axes for aesthetics.

## Reflection *(write 2–3 sentences in this cell)*

Before you submit, answer the three reflection prompts from [`09-project-task.md`](../09-project-task.md) right here:

1. What was the most confusing part of getting Colab + GPU + PyTorch set up? How did you resolve it?
2. Pick one galaxy class (E / S / S0 / Irr) and describe what features a CNN might need to recognise it.
3. Why do you think we'll spend an entire week on data pipelines next, before any model training?

 ans of 2: Spiral Galaxy (S)

A CNN would need to recognize:

Spiral arms extending outward from the center.
A bright central bulge.
Curved and symmetric patterns around the core.
Blue star-forming regions often found in the spiral arms.
The overall disk-like structure of the galaxy.

The CNN would learn these features automatically by detecting edges, curves, textures, and brightness patterns in different layers of the network.


ans of 3: Data pipelines are important because a neural network is only as good as the data it receives. Before training a model, we need to ensure that images are loaded correctly, resized to a consistent shape, converted into tensors, normalized properly, labeled correctly, and organized into batches